# Bloque Base: correlaciones entre candidatas

Qué variables de las 27 candidatas a Base dicen casi lo mismo. Solo mira las variables entre sí: **no usa la mortalidad**, así que no dice cuál predice mejor.

Tres miradas:
1. **Parejas:** correlación de cada par (Pearson y Spearman).
2. **Fórmulas:** variables que se calculan a partir de otras. Dos variables pueden ser redundantes sin que su correlación por parejas sea alta.
3. **R² frente al resto:** cuánto de cada variable se puede deducir de las otras 26 juntas.

Notebook de trabajo: importa de `src/`. Usa todo Gold, porque el split aún no está hecho.

In [1]:
# Imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.config.paths import GOLD_TABLE

In [2]:
# Base candidates
BASE_CANDIDATES = [
    "age", "sex", "weight", "height",
    "erythrocytes", "hemoglobin", "hematocrit", "mcv", "mch", "mchc", "rdw",
    "leukocytes", "neutrophils", "lymphocytes", "monocytes", "eosinophils", "basophils",
    "platelets", "mpv",
    "glucose", "creatinine", "urea", "uric_acid", "total_cholesterol", "hdl", "ast", "alt",
]

# Reading thresholds
HIGH = 0.7
MODERATE = 0.5

gold = pd.read_parquet(GOLD_TABLE)
base = gold[BASE_CANDIDATES].assign(sex=(gold["sex"] == "male").astype(float))
print(f"{base.shape[1]} candidates, {len(base)} people, {len(base.dropna())} with all of them")

27 candidates, 48495 people, 46697 with all of them


## 1. Parejas

Se ordena por la mayor de las dos correlaciones en valor absoluto. Spearman capta relaciones monótonas aunque no sean rectas, y aguanta mejor los valores extremos. `sex` va como 1 = hombre.

In [3]:
def correlated_pairs(data, threshold):
    """Pairs with |Pearson| or |Spearman| at or above the threshold, strongest first."""
    pearson, spearman = data.corr(), data.corr(method="spearman")
    rows = [
        {"variable_a": a, "variable_b": b, "pearson": pearson.loc[a, b], "spearman": spearman.loc[a, b]}
        for i, a in enumerate(data.columns) for b in data.columns[i + 1:]
    ]
    pairs = pd.DataFrame(rows)
    pairs["strongest"] = pairs[["pearson", "spearman"]].abs().max(axis=1)
    return pairs[pairs["strongest"] >= threshold].sort_values("strongest", ascending=False).drop(columns="strongest")


pairs = correlated_pairs(base, MODERATE)
print(f"Pairs out of {len(BASE_CANDIDATES) * (len(BASE_CANDIDATES) - 1) // 2}: {len(pairs)} at |r| >= {MODERATE}")
pairs.round(2).reset_index(drop=True)

Pairs out of 351: 15 at |r| >= 0.5


,variable_a,variable_b,pearson,spearman
0,hemoglobin,hematocrit,0.96,0.96
1,mcv,mch,0.94,0.90
2,leukocytes,neutrophils,0.74,0.91
3,erythrocytes,hematocrit,0.80,0.82
4,leukocytes,lymphocytes,0.77,0.56
5,erythrocytes,hemoglobin,0.72,0.76
6,ast,alt,0.75,0.73
7,sex,height,0.67,0.70
8,mch,mchc,0.64,0.57
9,sex,hemoglobin,0.57,0.61


## 2. Fórmulas

Se comprueba si cada variable coincide con la fórmula que la define. Una correlación cercana a 1 significa que esa columna no aporta un dato nuevo, solo otra forma de escribir los mismos.

In [4]:
formulas = {
    "leukocytes = neutrophils + lymphocytes + monocytes + eosinophils + basophils":
        (base["leukocytes"], base[["neutrophils", "lymphocytes", "monocytes", "eosinophils", "basophils"]].sum(axis=1, min_count=5)),
    "mcv = hematocrit / erythrocytes":  (base["mcv"], base["hematocrit"] / base["erythrocytes"]),
    "mch = hemoglobin / erythrocytes":  (base["mch"], base["hemoglobin"] / base["erythrocytes"]),
    "mchc = hemoglobin / hematocrit":   (base["mchc"], base["hemoglobin"] / base["hematocrit"]),
}
pd.Series({name: left.corr(right) for name, (left, right) in formulas.items()}, name="pearson").round(4).to_frame()

,pearson
leukocytes = neutrophils + lymphocytes + monocytes + eosinophils + basophils,0.9997
mcv = hematocrit / erythrocytes,0.9990
mch = hemoglobin / erythrocytes,0.9988
mchc = hemoglobin / hematocrit,0.9934


## 3. R² frente al resto

Regresión lineal de cada variable sobre las otras 26, con las personas que tienen todas. Un R² de 0,90 significa que el 90 % de esa variable se puede deducir de las demás. Es lineal, así que las fórmulas con cocientes pueden quedar algo por debajo de 1.

In [5]:
def r2_against_rest(data):
    """R2 of a linear regression of each column on all the others."""
    complete = data.dropna()
    scores = {}
    for column in complete.columns:
        rest = complete.drop(columns=column).to_numpy()
        design = np.column_stack([np.ones(len(rest)), rest])
        target = complete[column].to_numpy()
        fitted = design @ np.linalg.lstsq(design, target, rcond=None)[0]
        scores[column] = 1 - ((target - fitted) ** 2).sum() / ((target - target.mean()) ** 2).sum()
    return pd.Series(scores, name="r2").sort_values(ascending=False)


r2_against_rest(base).round(3).to_frame()

,r2
leukocytes,0.999
lymphocytes,0.999
hematocrit,0.999
mch,0.999
neutrophils,0.999
hemoglobin,0.998
mcv,0.998
erythrocytes,0.992
mchc,0.989
monocytes,0.897


## 4. Serie roja: qué se recupera con cada par

Para cada par de variables que se conserva, R² de cada una de las demás a partir de ese par, con términos lineales y cuadráticos. Un 1,00 significa que no se pierde nada; cuanto más bajo, más información propia se pierde al quitarla.

In [6]:
RED_CELLS = ["hemoglobin", "hematocrit", "erythrocytes", "mcv", "mch", "mchc"]
KEPT_PAIRS = [
    ("hemoglobin", "erythrocytes"), ("hemoglobin", "hematocrit"), ("hemoglobin", "mcv"),
    ("hematocrit", "erythrocytes"), ("erythrocytes", "mcv"),
]


def r2_from_kept(data, kept, target):
    """R2 of target from a pair of kept columns, with squares and their interaction."""
    a, b = data[kept[0]].to_numpy(), data[kept[1]].to_numpy()
    design = np.column_stack([np.ones(len(a)), a, b, a ** 2, b ** 2, a * b])
    values = data[target].to_numpy()
    fitted = design @ np.linalg.lstsq(design, values, rcond=None)[0]
    return 1 - ((values - fitted) ** 2).sum() / ((values - values.mean()) ** 2).sum()


red = base[RED_CELLS].dropna()
pd.DataFrame(
    {" + ".join(kept): {t: r2_from_kept(red, kept, t) for t in RED_CELLS if t not in kept} for kept in KEPT_PAIRS}
).round(3)

,hemoglobin + erythrocytes,hemoglobin + hematocrit,hemoglobin + mcv,hematocrit + erythrocytes,erythrocytes + mcv
hematocrit,0.953,NaN,0.936,NaN,0.999
mcv,0.882,0.245,NaN,0.997,NaN
mch,0.997,0.482,0.892,0.884,0.887
mchc,0.417,0.986,0.207,0.166,0.170
erythrocytes,NaN,0.703,0.939,NaN,NaN
hemoglobin,NaN,NaN,NaN,0.940,0.940
